In [2]:
# ============================================================
# السيل 1 — الإعداد والتثبيت + الإعدادات المجمّدة (prompt_version = 1)
# ============================================================
# (أ) SDK الرسمي الجديد من Google (لا المهجور google-generativeai)
!pip install -q -U google-genai

# (ب) الاستيرادات
import os, json, csv, time, random, mimetypes, traceback
from datetime import datetime, timezone
from collections import defaultdict, Counter
from importlib.metadata import version as _pkgver

from google import genai
from google.genai import types

print("✅ google-genai مثبّت — الإصدار:", _pkgver("google-genai"))

# (ج) الإعدادات المجمّدة — تُستخدم لاحقًا في حلقة التشغيل (السيلز 6-8)،
#     ومثبّتة الآن كأساس ثابت لكل النماذج الثمانية (prompt_version = 1).
MODEL_NAME        = "gemini-3.5-flash"   # ← القرار النهائي (يدعم إطفاء التفكير) # موديل العيّنة التجريبية (Free Tier)
TEMPERATURE       = 0                     # حتمية قدر الإمكان -> نتائج قابلة للتكرار
PROMPT_VERSION    = 1                     # رقم بروتوكول البرومبت المجمّد
MAX_OUTPUT_TOKENS = 64                    # يكفي مع إطفاء التفكير (الخرج = 2 توكن)
THINKING_BUDGET   = 0                     # إطفاء "تفكير" 2.5-flash: أسرع وأرخص،
FALLBACK_MAX_TOKENS = 512                   # ← جديد: لنموذج يرفض الإطفاء (مساحة للتفكير +

# بروتوكول base المجمّد: الصورة + السؤال فقط، مع تعليمة إخراج صارمة (لا سياق، لا تلميح).
SYSTEM_INSTRUCTION_AR = (
    "انظر إلى الصورة وأجب عن السؤال بالاعتماد على ما هو ظاهر فيها. "
    "يجب أن تكون إجابتك كلمة واحدة فقط: نعم أو لا. "
    "لا تضف أي شرح أو علامات ترقيم أو كلمات أخرى."
)

print("✅ الإعدادات المجمّدة جاهزة | الموديل:", MODEL_NAME,
      "| temperature:", TEMPERATURE, "| prompt_version:", PROMPT_VERSION)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 10.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.0 which is incompatible.
✅ google-genai مثبّت — الإصدار: 2.19.0
✅ الإعدادات المجمّدة جاهزة | الموديل: gemini-3.5-flash | temperature: 0 | prompt_version: 1


In [3]:
# ============================================================
# السيل 2 — ربط Google Drive + PROJECT_ROOT + التحقق الاستباقي الكامل
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT   = "/content/drive/MyDrive/Datasets-AraPhD"
QUESTIONS_FILE = os.path.join(PROJECT_ROOT, "questions_base_mahyoub.json")
POOL_FILE      = os.path.join(PROJECT_ROOT, "image_pool_mahyoub.json")
RESULTS_DIR    = os.path.join(PROJECT_ROOT, "results_mahyoub")   # مخرجات السيلز 6-8

problems = []
if not os.path.isdir(PROJECT_ROOT):
    problems.append(f"مجلد المشروع غير موجود: {PROJECT_ROOT}")
    mydrive = "/content/drive/MyDrive"
    if os.path.isdir(mydrive):
        print("📂 المجلدات تحت MyDrive (لضبط PROJECT_ROOT إن اختلف الاسم):")
        for name in sorted(os.listdir(mydrive))[:50]:
            print("   -", name)
else:
    for f in (QUESTIONS_FILE, POOL_FILE):
        if not os.path.isfile(f):
            problems.append(f"ملف مفقود: {f}")

if problems:
    print("\n❌ توقف — أصلح ما يلي ثم أعد تشغيل هذا السيل:")
    for p in problems: print("   •", p)
    raise RuntimeError("فشل التحقق الأساسي — راجع القائمة أعلاه.")

# تحميل الملفين مرة واحدة (يُعاد استخدامهما في السيل 4) + خريطة image_id -> مسار الصورة
with open(QUESTIONS_FILE, encoding='utf-8') as fh:
    QUESTIONS = json.load(fh)['items']
with open(POOL_FILE, encoding='utf-8') as fh:
    POOL = json.load(fh)
ID2PATH = {r['image_id']: os.path.join(PROJECT_ROOT, r['local_path']) for r in POOL}

# التحقق من ربط كل سؤال بصورة موجودة فعلًا
unresolved, missing_files, empty_files = [], [], []
for it in QUESTIONS:
    iid = it['image_id']
    if iid not in ID2PATH:
        unresolved.append(iid); continue
    p = ID2PATH[iid]
    if not os.path.isfile(p):        missing_files.append((iid, p))
    elif os.path.getsize(p) == 0:    empty_files.append((iid, p))

print("\n===== تقرير التحقق الاستباقي =====")
print(f"عدد الأسئلة (صور): {len(QUESTIONS)}")
print(f"لها image_id في المسبح: {len(QUESTIONS) - len(unresolved)}/{len(QUESTIONS)}")
ok = len(QUESTIONS) - len(unresolved) - len(missing_files) - len(empty_files)
print(f"صور موجودة فعلًا على Drive: {ok}/{len(QUESTIONS)}")
for label, lst in [("بلا image_id في المسبح", unresolved),
                   ("ملف صورة مفقود", missing_files),
                   ("ملف صورة فارغ (0 بايت)", empty_files)]:
    if lst:
        print(f"\n⚠️ {label}: {len(lst)} — أمثلة:")
        for x in lst[:10]: print("   ", x)

if unresolved or missing_files or empty_files:
    raise RuntimeError("فشل التحقق — بعض الصور غير موجودة/فارغة. صحّح المسارات قبل المتابعة.")

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"\n✅ كل الملفات و{len(QUESTIONS)} صورة موجودة. مجلد المخرجات جاهز: {RESULTS_DIR}")
print("✅ التحقق الاستباقي ناجح — تابع للسيل 3.")

Mounted at /content/drive

===== تقرير التحقق الاستباقي =====
عدد الأسئلة (صور): 270
لها image_id في المسبح: 270/270
صور موجودة فعلًا على Drive: 270/270

✅ كل الملفات و270 صورة موجودة. مجلد المخرجات جاهز: /content/drive/MyDrive/Datasets-AraPhD/results_mahyoub
✅ التحقق الاستباقي ناجح — تابع للسيل 3.


In [4]:
# ============================================================
# السيل 3 — تحميل مفتاح Gemini API من Colab Secrets + إنشاء العميل
# ============================================================
from google.colab import userdata

try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception as e:
    GEMINI_API_KEY = None
    print("⚠️ تعذّر قراءة Secrets:", e)

if not GEMINI_API_KEY:
    raise RuntimeError(
        "لم يُعثر على GEMINI_API_KEY.\n"
        "في Colab: أيقونة المفتاح 🔑 (يسار) → Add new secret →\n"
        "الاسم: GEMINI_API_KEY، القيمة: مفتاحك، وفعّل (Notebook access)."
    )

client = genai.Client(api_key=GEMINI_API_KEY)   # العميل = قناة الطلبات إلى Gemini
print("✅ المفتاح مقروء والعميل جاهز. طول المفتاح:", len(GEMINI_API_KEY), "حرفًا (لا نطبع المفتاح).")

# فحص مصادقة خفيف (نداء ميتاداتا مجاني، لا يمسّ حصّة التوليد)
try:
    names = [m.name for m in client.models.list()]
    has_model = any("2.5-flash" in (n or "") for n in names)
    print(f"✅ المصادقة نجحت — موديلات متاحة: {len(names)}")
    print("   هل", MODEL_NAME, "متاح؟", "نعم" if has_model else "لم يظهر بالاسم المتوقع (سنتأكد في السيل 6)")
except Exception as e:
    print("⚠️ تعذّر فحص المصادقة الآن:", repr(e))
    print("   إن كان خطأ مصادقة/صلاحية → أوقف وصحّح المفتاح. إن كان عابرًا → الاختبار الحقيقي في السيل 6.")

✅ المفتاح مقروء والعميل جاهز. طول المفتاح: 53 حرفًا (لا نطبع المفتاح).
✅ المصادقة نجحت — موديلات متاحة: 50
   هل gemini-3.5-flash متاح؟ نعم


In [5]:
# ============================================================
# السيل 4 — بناء قائمة المهام (540) — كل مهمة = صورة + سؤال + الجواب الصحيح
# ============================================================
for _v in ("QUESTIONS", "ID2PATH"):
    if _v not in globals():
        raise RuntimeError(f"المتغيّر {_v} غير موجود — شغّل السيل 2 أولًا.")

TASKS = []
for it in QUESTIONS:
    iid  = it['image_id']
    base = dict(image_id=iid, filename=it['filename'], image_path=ID2PATH[iid],
                category=it['category'], hitem_level=it['hitem_level'],
                gt_ar=it['gt_ar'], h_item_ar=it['h_item_ar'])
    # سؤال الهوية الصحيحة -> "نعم"
    TASKS.append({**base, "task_id": f"{iid}__yes", "polarity": "yes",
                  "question_text": it['question_yes_ar'], "ground_truth": it['answer_yes']})
    # سؤال البديل المضلِّل -> "لا"
    TASKS.append({**base, "task_id": f"{iid}__no", "polarity": "no",
                  "question_text": it['question_no_ar'], "ground_truth": it['answer_no']})

assert len(TASKS) == 540, f"عدد المهام {len(TASKS)} ≠ 540"
assert len({t['task_id'] for t in TASKS}) == 540, "task_id غير فريد!"
print("✅ قائمة المهام جاهزة:", len(TASKS), "مهمة")
print("   القطبية:  ", dict(Counter(t['polarity'] for t in TASKS)))
print("   الفئات:   ", dict(Counter(t['category'] for t in TASKS)))
print("   hitem_level:", dict(Counter(t['hitem_level'] for t in TASKS)))
print("\nمثال (نفس الصورة، سؤالان):")
for t in TASKS[:2]:
    print(f"   [{t['polarity']}] {t['question_text']}  → الصحيح: {t['ground_truth']}")

✅ قائمة المهام جاهزة: 540 مهمة
   القطبية:   {'yes': 270, 'no': 270}
   الفئات:    {'cuisine': 386, 'objects': 54, 'architecture': 20, 'attire': 80}
   hitem_level: {'identity': 468, 'attribute': 72}

مثال (نفس الصورة، سؤالان):
   [yes] هل هذا الطبق صينية بطاطا؟  → الصحيح: نعم
   [no] هل هذا الطبق شية؟  → الصحيح: لا


In [6]:
# ============================================================
# السيل 5 — عيّنة متوازنة: 10 صور (تغطي الفئات الأربع) = 20 سؤالًا
# ============================================================
if "TASKS" not in globals():
    raise RuntimeError("TASKS غير موجودة — شغّل السيل 4 أولًا.")

SAMPLE_SEED = 42
TARGET = {"cuisine": 4, "attire": 2, "objects": 2, "architecture": 2}   # 10 صور = 20 سؤالًا
random.seed(SAMPLE_SEED)

by_cat = defaultdict(list)
for it in QUESTIONS:
    by_cat[it['category']].append(it)
for cat, n in TARGET.items():
    if len(by_cat[cat]) < n:
        raise RuntimeError(f"الفئة {cat} فيها {len(by_cat[cat])} صورة فقط < {n}")

selected = []
for cat, n in TARGET.items():
    pool_cat = by_cat[cat][:]
    random.shuffle(pool_cat)
    selected.extend(pool_cat[:n])

# ضمان صورة attribute واحدة على الأقل (نفضّل استبدال cuisine-identity بـ attribute)
sel_ids = {it['image_id'] for it in selected}
if not any(it['hitem_level'] == 'attribute' for it in selected):
    attr_imgs = [it for it in QUESTIONS
                 if it['hitem_level'] == 'attribute' and it['image_id'] not in sel_ids]
    attr_imgs.sort(key=lambda x: (x['category'] != 'cuisine', x['image_id']))
    if attr_imgs:
        repl = attr_imgs[0]
        for i, it in enumerate(selected):
            if it['category'] == 'cuisine' and it['hitem_level'] == 'identity':
                selected[i] = repl; break

sel_ids = {it['image_id'] for it in selected}
SAMPLE_TASKS = [t for t in TASKS if t['image_id'] in sel_ids]

assert len(selected) == 10 and len(SAMPLE_TASKS) == 20
assert any(it['hitem_level'] == 'attribute' for it in selected), "لا توجد صورة attribute!"
print("✅ عيّنة الاختبار:", len(SAMPLE_TASKS), "مهمة من", len(selected), "صور")
print("   الفئات (صور):  ", dict(Counter(it['category'] for it in selected)))
print("   hitem_level:   ", dict(Counter(it['hitem_level'] for it in selected)))
print("   القطبية (مهام):", dict(Counter(t['polarity'] for t in SAMPLE_TASKS)))
print("\nالصور العشر المختارة:")
for it in selected:
    print(f"   {it['image_id']:<24} | {it['category']:<12} | {it['hitem_level']:<9} | "
          f"gt={it['gt_ar']} | h={it['h_item_ar']}")

✅ عيّنة الاختبار: 20 مهمة من 10 صور
   الفئات (صور):   {'cuisine': 4, 'attire': 2, 'objects': 2, 'architecture': 2}
   hitem_level:    {'attribute': 2, 'identity': 8}
   القطبية (مهام): {'yes': 10, 'no': 10}

الصور العشر المختارة:
   image_515                | cuisine      | attribute | gt=بلح أخضر | h=جاهزة للأكل بحلاوتها الكاملة في هذه المرحلة
   image_148A;image_148B    | cuisine      | identity  | gt=سلطة برغل | h=كسكسي
   image_104A;image_104B    | cuisine      | identity  | gt=كشري | h=مجدرة
   image_277A;image_277B    | cuisine      | identity  | gt=قطايف | h=حلاوة الجبن
   image_492                | attire       | identity  | gt=عباية | h=جلابة
   image_497                | attire       | identity  | gt=الثوب الفلسطيني المطرز | h=جلابة
   image_545                | objects      | identity  | gt=دف | h=طبلة أسطوانية صغيرة
   image_161A;image_161B    | objects      | attribute | gt=قفة | h=تُستخدم تقليديًا لتخزين الملابس
   image_267A;image_267B    | architecture | identity  | gt

In [7]:
# ============================================================
# السيل 6 — المحرّك: نداء Gemini + محلِّل نعم/لا/unclear + طبقات الحماية
# ============================================================
import re
from google.genai import errors

# --- إعدادات الإيقاع (Free Tier) ---
MIN_GAP_SECONDS = 15.0    # كان 8.0 — انتظار أطول = أمان أكثر    # فجوة آمنة بين كل طلب وآخر (تبقينا تحت حدّ الدقيقة)
MAX_RETRIES     = 5       # سقف المحاولات لكل مهمة (ثم نسجّلها خطأ وننتقل)

if "client" not in globals():
    raise RuntimeError("العميل غير موجود — شغّل السيل 3 أولًا.")

FIELDNAMES = ["task_id","image_id","filename","category","polarity","hitem_level",
              "gt_ar","h_item_ar","question_text","ground_truth","raw_response",
              "parsed_answer","is_correct","error_type","detail","attempts",
              "latency_s","model_name","thinking_disabled","prompt_version",
              "temperature","timestamp"]

_THINK_OFF_OK = {}          # ذاكرة: هل يقبل هذا الموديل إطفاء التفكير؟

def _build_config(model):
    """إطفاء التفكير حيث يُدعم؛ وإلا مساحة أوسع كي لا يبتلع التفكيرُ الجواب."""
    if _THINK_OFF_OK.get(model, True):
        return types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION_AR, temperature=TEMPERATURE,
            max_output_tokens=MAX_OUTPUT_TOKENS,
            thinking_config=types.ThinkingConfig(thinking_budget=THINKING_BUDGET)), True
    return types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION_AR, temperature=TEMPERATURE,
        max_output_tokens=FALLBACK_MAX_TOKENS), False

def _mk_result(task, raw, parsed, error_type, detail, attempts, latency, think_off=True):
    gt = task["ground_truth"]
    is_correct = int((parsed=="yes" and gt=="نعم") or (parsed=="no" and gt=="لا"))
    return {"task_id":task["task_id"], "image_id":task["image_id"], "filename":task["filename"],
            "category":task["category"], "polarity":task["polarity"], "hitem_level":task["hitem_level"],
            "gt_ar":task["gt_ar"], "h_item_ar":task["h_item_ar"], "question_text":task["question_text"],
            "ground_truth":gt, "raw_response":raw, "parsed_answer":parsed, "is_correct":is_correct,
            "error_type":error_type, "detail":detail, "attempts":attempts, "latency_s":latency,
            "model_name":MODEL_NAME, "thinking_disabled":int(think_off),
            "prompt_version":PROMPT_VERSION, "temperature":TEMPERATURE,
            "timestamp":datetime.now(timezone.utc).isoformat()}

def run_one_task(task):
    try:
        with open(task["image_path"], "rb") as fh: img_bytes = fh.read()
        img_part = types.Part.from_bytes(data=img_bytes, mime_type=_guess_mime(task["image_path"]))
    except Exception as e:
        return _mk_result(task, "", "unclear", "image_load_error", str(e)[:200], 0, 0.0)

    for attempt in range(1, MAX_RETRIES + 1):
        cfg, think_off = _build_config(MODEL_NAME)
        _pace(); t0 = time.time()
        try:
            resp = client.models.generate_content(
                model=MODEL_NAME, contents=[img_part, task["question_text"]], config=cfg)
            latency = round(time.time() - t0, 2)
            try: raw = (resp.text or "").strip()
            except Exception: raw = ""
            if not raw:
                fr = _finish_reason(resp)
                blocked = fr and any(k in fr.upper() for k in ("SAFETY","PROHIBITED","BLOCK","RECITATION"))
                # فارغ بسبب MAX_TOKENS مع تفكير مُفعَّل -> وسّع المساحة وأعد
                if (not blocked) and (fr and "MAX_TOKENS" in fr.upper()) and think_off is False \
                   and attempt < MAX_RETRIES:
                    continue
                return _mk_result(task, "", "unclear",
                                  "safety_block" if blocked else "empty_response",
                                  fr or "", attempt, latency, think_off)
            return _mk_result(task, raw, parse_answer(raw), "", "", attempt, latency, think_off)

        except errors.APIError as e:
            code = getattr(e, "code", None); msg = str(getattr(e, "message", e))
            # الموديل يرفض إطفاء التفكير -> تحوّل تلقائي مرة واحدة ويُحفظ
            if code == 400 and "thinking" in msg.lower() and _THINK_OFF_OK.get(MODEL_NAME, True):
                _THINK_OFF_OK[MODEL_NAME] = False
                print(f"   ↪ {MODEL_NAME}: لا يدعم إطفاء التفكير — تحوّل إلى {FALLBACK_MAX_TOKENS} توكن")
                continue
            transient = (code == 429) or (isinstance(code, int) and 500 <= code < 600)
            if transient and attempt < MAX_RETRIES:
                time.sleep(_backoff(attempt, e)); continue
            et = "rate_limited" if code == 429 else (
                 f"server_error_{code}" if isinstance(code,int) and code>=500 else f"api_error_{code}")
            return _mk_result(task, "", "unclear", et, msg[:200], attempt,
                              round(time.time()-t0,2), think_off)
        except Exception as e:
            if attempt < MAX_RETRIES:
                time.sleep(_backoff(attempt, None)); continue
            return _mk_result(task, "", "unclear", "exception", str(e)[:200], attempt,
                              round(time.time()-t0,2), think_off)

# اختبار نداء واحد
_demo = run_one_task(SAMPLE_TASKS[0])
print("سؤال:", SAMPLE_TASKS[0]["question_text"])
print("الصحيح:", _demo["ground_truth"], "| الخام:", repr(_demo["raw_response"]))
print("المُستخرج:", _demo["parsed_answer"], "| صحيح؟", _demo["is_correct"],
      "| تفكير مُطفأ؟", _demo["thinking_disabled"], "| زمن:", _demo["latency_s"], "ث")

سؤال: هل هذا الطبق كشري؟
الصحيح: نعم | الخام: ''
المُستخرج: unclear | صحيح؟ 0 | تفكير مُطفأ؟ 1 | زمن: 0.0 ث


In [8]:
# ============================================================
# السيل 7 — حلقة التشغيل (افتراضيًا: العيّنة 20) + حفظ دوري + استئناف آمن
# ============================================================
RUN_LABEL   = "sample"                                   # ← لاحقًا: غيّرها إلى "full" للـ540
RUN_TASKS   = SAMPLE_TASKS if RUN_LABEL == "sample" else TASKS
MODEL_SLUG  = re.sub(r'[^a-z0-9.]+', '-', MODEL_NAME.lower().replace("models/", ""))
RESULTS_CSV = os.path.join(RESULTS_DIR, f"results_base_{RUN_LABEL}_{MODEL_SLUG}.csv")
# الأخطاء الدائمة لا تُعاد؛ الأخطاء العابرة (rate_limited...) تُعاد عند الاستئناف
PERMANENT = {"image_load_error","safety_block","api_error_400","api_error_403"}
def _is_final(row):
    return row.get("parsed_answer") in ("yes","no") or row.get("error_type") in PERMANENT

done = set()                                             # استئناف: ماذا اكتمل نهائيًا؟
if os.path.isfile(RESULTS_CSV):
    with open(RESULTS_CSV, newline="", encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            if _is_final(row): done.add(row["task_id"])

todo = [t for t in RUN_TASKS if t["task_id"] not in done]
print(f"التشغيل: {RUN_LABEL} | الإجمالي: {len(RUN_TASKS)} | مكتمل سابقًا: {len(done)} | متبقٍّ: {len(todo)}")
if not todo:
    print("✅ لا شيء متبقٍّ — كل المهام مكتملة. انتقل للسيل 8.");
else:
    new_file = not os.path.isfile(RESULTS_CSV)
    consec_quota, saved = 0, 0
    with open(RESULTS_CSV, "a", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=FIELDNAMES)
        if new_file: writer.writeheader()
        for i, task in enumerate(todo, 1):
            res = run_one_task(task)
            writer.writerow(res); fh.flush()             # حفظ فوري على القرص
            saved += 1
            status = res["parsed_answer"] if not res["error_type"] else f"⚠️{res['error_type']}"
            print(f"[{i}/{len(todo)}] {task['task_id'][:26]:<26} {task['polarity']:<3} → {status}")
            if res["error_type"] == "rate_limited":      # حارس الحصّة اليومية
                consec_quota += 1
                if consec_quota >= 3:
                    print("\n⛔ يبدو أن الحصّة نفدت (اليومية غالبًا). إيقاف نظيف — النتائج محفوظة.")
                    print("   أعد تشغيل هذا السيل لاحقًا (غدًا) وسيُكمل من حيث توقّف تلقائيًا.")
                    break
            else:
                consec_quota = 0
    print(f"\n✅ حُفظت {saved} نتيجة في: {RESULTS_CSV}")
    print("انتقل للسيل 8 لعرض المقاييس.")

التشغيل: sample | الإجمالي: 20 | مكتمل سابقًا: 16 | متبقٍّ: 4
[1/4] image_515__yes             yes → ⚠️image_load_error
[2/4] image_515__no              no  → ⚠️image_load_error
[3/4] image_545__yes             yes → ⚠️image_load_error
[4/4] image_545__no              no  → ⚠️image_load_error

✅ حُفظت 4 نتيجة في: /content/drive/MyDrive/Datasets-AraPhD/results_mahyoub/results_base_sample_gemini-3.5-flash.csv
انتقل للسيل 8 لعرض المقاييس.


In [9]:
# ============================================================
# السيل 8 — المقاييس + التفصيل (اقرأ نفس RESULTS_CSV من السيل 7)
# ============================================================
rows_by_task = {}
with open(RESULTS_CSV, newline="", encoding="utf-8") as fh:
    for row in csv.DictReader(fh): rows_by_task[row["task_id"]] = row   # آخر نتيجة تفوز
rows = list(rows_by_task.values())
def _b(x): return str(x) == "1"

total   = len(rows)
parsed  = [r for r in rows if r["parsed_answer"] in ("yes","no")]
yes_gt  = [r for r in rows if r["ground_truth"] == "نعم"]
no_gt   = [r for r in rows if r["ground_truth"] == "لا"]
acc_all    = sum(_b(r["is_correct"]) for r in rows)/total if total else 0
acc_parsed = sum(_b(r["is_correct"]) for r in parsed)/len(parsed) if parsed else 0
yes_recall = sum(_b(r["is_correct"]) for r in yes_gt)/len(yes_gt) if yes_gt else 0
no_recall  = sum(_b(r["is_correct"]) for r in no_gt)/len(no_gt) if no_gt else 0
yes_rate   = sum(1 for r in parsed if r["parsed_answer"]=="yes")/len(parsed) if parsed else 0
by_img = defaultdict(dict)
for r in rows: by_img[r["image_id"]][r["polarity"]] = _b(r["is_correct"])
paired = [d for d in by_img.values() if "yes" in d and "no" in d]
paired_acc = sum(1 for d in paired if d["yes"] and d["no"])/len(paired) if paired else 0

print(f"========== نتائج {os.path.basename(RESULTS_CSV)} ==========")
print(f"المهام: {total} | مُصنّفة: {len(parsed)} | غير واضحة/أخطاء: {total-len(parsed)} | نسبة الوضوح: {len(parsed)/total:.1%}")
print(f"الدقة الكلية (unclear=خطأ): {acc_all:.1%}")
print(f"الدقة على المُصنّفة فقط:    {acc_parsed:.1%}")
print(f"Yes-Recall: {yes_recall:.1%} | No-Recall: {no_recall:.1%}")
print(f"انحياز 'نعم' (50%=محايد): {yes_rate:.1%}")
print(f"الدقة المزدوجة (السؤالان معًا): {paired_acc:.1%} على {len(paired)} صورة")

def breakdown(key):
    agg = defaultdict(lambda: [0,0])
    for r in rows:
        a = agg[r[key]]; a[0]+= _b(r["is_correct"]); a[1]+=1
    return {k:(c/n, n) for k,(c,n) in sorted(agg.items())}
print("\nحسب الفئة:")
for k,(acc,n) in breakdown("category").items():    print(f"   {k:<13} {acc:.1%}  (n={n})")
print("حسب hitem_level:")
for k,(acc,n) in breakdown("hitem_level").items(): print(f"   {k:<10} {acc:.1%}  (n={n})")

errs = Counter(r["error_type"] for r in rows if r["error_type"])
if errs: print("\nأنواع الأخطاء:", dict(errs))
print("\n— الإجابات الخاطئة/غير الواضحة (لمراجعتك) —")
for r in rows:
    if not _b(r["is_correct"]):
        tag = f"⚠️{r['error_type']}" if r["error_type"] else ""
        print(f"   [{r['polarity']}] {r['question_text']} | صحيح={r['ground_truth']} | ردّ={r['parsed_answer']} «{r['raw_response'][:25]}» {tag}")

========== نتائج results_base_sample_gemini-3.5-flash.csv ==========
المهام: 20 | مُصنّفة: 16 | غير واضحة/أخطاء: 4 | نسبة الوضوح: 80.0%
الدقة الكلية (unclear=خطأ): 75.0%
الدقة على المُصنّفة فقط:    93.8%
Yes-Recall: 80.0% | No-Recall: 70.0%
انحياز 'نعم' (50%=محايد): 56.2%
الدقة المزدوجة (السؤالان معًا): 70.0% على 10 صورة

حسب الفئة:
   architecture  100.0%  (n=4)
   attire        100.0%  (n=4)
   cuisine       62.5%  (n=8)
   objects       50.0%  (n=4)
حسب hitem_level:
   attribute  50.0%  (n=4)
   identity   81.2%  (n=16)

أنواع الأخطاء: {'image_load_error': 4}

— الإجابات الخاطئة/غير الواضحة (لمراجعتك) —
   [no] هل هذا الطبق كسكسي؟ | صحيح=لا | ردّ=yes «نعم» 
   [yes] هل تحتاج هذه الثمار مراحل نضج إضافية قبل أن تكتسب حلاوتها؟ | صحيح=نعم | ردّ=unclear «» ⚠️image_load_error
   [no] هل هذه الثمار جاهزة للأكل بحلاوتها الكاملة في هذه المرحلة؟ | صحيح=لا | ردّ=unclear «» ⚠️image_load_error
   [yes] هل هذه الآلة دف؟ | صحيح=نعم | ردّ=unclear «» ⚠️image_load_error
   [no] هل هذه الآلة طبلة أسطو

In [11]:
with open(RESULTS_CSV, newline="", encoding="utf-8") as fh:
    for row in csv.DictReader(fh):
        if row["error_type"] == "rate_limited":
            print(row["detail"])
            print("---")
            print(len(row["detail"]), "حرف")
            break

You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usa
---
200 حرف
